# eBird API
## MAD 2502 — Applied Data

**Big idea:** An API lets one program request structured data from another service.

By the end, you should be able to explain an endpoint, header, parameter, and JSON response; turn API records into a DataFrame; distinguish rows from counts; and reuse one request for another species or region.

> **Lesson rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect


[View on GitHub](https://github.com/mathguy-uf/MAD2502/blob/main/applied_data/01_ebird_api.ipynb) · [Run in Google Colab](https://colab.research.google.com/github/mathguy-uf/MAD2502/blob/main/applied_data/01_ebird_api.ipynb) · [Launch Binder/JupyterLab](https://mybinder.org/v2/gh/mathguy-uf/MAD2502/main?urlpath=lab/tree/applied_data/01_ebird_api.ipynb)

Run cells from the top with **Shift+Enter**. Restart the kernel and run all to check your work. Each tutorial includes its own imports and data. Hosted sessions are temporary: download your work before leaving.

In [ ]:
# If an import fails, uncomment and run this once, then restart the kernel.
# %pip install pandas requests

## 1. Understand — asking another program for data
An **API** (application programming interface) is a documented way for programs to communicate. A **request** asks a service for something; its **response** contains a status and usually data. An **endpoint** is the service address for a particular operation.

We use the recent-observations endpoint for one species:
```text
https://api.ebird.org/v2/data/obs/{region_code}/recent/{species_code}
```
A **region code** identifies a place: `US-FL` is Florida. A **species code** is eBird's identifier, not just any abbreviation: `norcar`, `blujay`, and `amecro` identify Northern Cardinal, Blue Jay, and American Crow.

An **API key** identifies your authorized access. **Headers** carry request metadata, including the key. **Parameters** select options: `back` is the number of days (1–30), and `maxResults` caps returned records. These recent records are a limited view of reported sightings, not all checklists, unique birds, or a population census.

**JSON** (JavaScript Object Notation) is language-independent text. A JSON array becomes a Python list; a JSON object becomes a dictionary; `null` becomes `None`. `response.json()` performs that conversion.

For optional live requests, copy `templates/my_api_info_example.py` to **my_api_info.py beside your notebook** and fill it locally:
```python
# my_api_info.py — never commit this file
ebird_api_key = "YOUR_KEY_HERE"
```
In Colab, upload that file privately to the session's Files pane. It disappears when the session ends. Do not share a notebook with key values in cells or outputs. `.gitignore` helps with Git; it does not encrypt files or protect a file that was already committed.

In [ ]:
import json
import pandas as pd
import requests
from IPython.display import display

RUN_LIVE = False  # Change to True only when you want a real request.
API_KEY = None
if RUN_LIVE:
    try:
        from my_api_info import ebird_api_key
        API_KEY = ebird_api_key
    except ImportError:
        print("No key file found. The sample lesson still works.")

## 2. Predict

Before running the next cell, predict the exact URL produced by these values:

```python
region_code = "US-FL"
species_code = "norcar"
days = 7
```

Also predict: Is `days` part of the URL path, the header, or the parameter dictionary?


In [ ]:
region_code = "US-FL"
species_code = "norcar"
days = 7

url = (
    "https://api.ebird.org/v2/data/obs/"
    f"{region_code}/recent/{species_code}"
)
params = {"back": days, "maxResults": 5000}
headers = {"X-eBirdApiToken": API_KEY}

print(url)
print(params)


## 3. Run

Start with an offline response. A common JSON response becomes a Python **list of dictionaries**, which maps naturally to a DataFrame.


In [ ]:
# Invented teaching records, not live eBird observations.
sample_json = """[
  {"speciesCode": "norcar", "locName": "Campus Garden", "howMany": 2},
  {"speciesCode": "norcar", "locName": "Lake Trail", "howMany": 5},
  {"speciesCode": "norcar", "locName": "Wetlands"}
]"""
practice_records = json.loads(sample_json)
print(type(practice_records), type(practice_records[0]))
print(practice_records[0]["locName"])
practice_birds = pd.DataFrame(practice_records)
display(practice_birds)
counts = pd.to_numeric(practice_birds["howMany"], errors="coerce")
print("returned records:", len(practice_birds))
print("known count total:", counts.sum(min_count=1))
print("missing counts:", counts.isna().sum())

Compare the output with what you expected. `len(df)` counts returned records; `sum()` adds known bird counts. Those answer different questions. A missing `howMany` means unknown, not automatically zero.


The third dictionary omits `howMany` entirely. pandas fills the gap with a missing value. `sum(min_count=1)` returns missing when every count is unknown, rather than incorrectly reporting zero. Multiple reports may refer to the same birds; a result cap may omit reports.

## 4. Change

Change **one thing at a time** and predict the effect before running it.

- Change `species_code` from `norcar` to `blujay`. What part of the URL changes?
- Change `days` from `7` to `10`. What part of the request changes?
- Add one more pretend record with `howMany = 12`. What happens to `len(...)` and to the sum?


In [ ]:
species_code = "blujay"
days = 10

changed_url = (
    "https://api.ebird.org/v2/data/obs/"
    f"{region_code}/recent/{species_code}"
)
changed_params = {"back": days, "maxResults": 5000}

print(changed_url)
print(changed_params)


## 5. Use — make the optional live request
An HTTP **status code** reports how the request went: 200 usually means success; 400 means a bad request; 401/403 often indicate authentication/access problems; 429 means too many requests; 5xx indicates a server problem. `raise_for_status()` raises an exception for HTTP error statuses before we try to analyze the body. A timeout prevents waiting forever.

The function below combines the pieces above. `if` checks an input, `raise` reports a problem, and `return` sends a table back. An empty JSON list can be a successful request with no matching records. We preserve named columns in that case. A missing `howMany` stays missing.

In [ ]:
def get_ebird_data(api_key, region_code, species_code, days=7, max_results=5000):
    """Return recent records; counts are not a bird population estimate."""
    if not api_key or api_key == "YOUR_KEY_HERE":
        raise ValueError("Add your own key in my_api_info.py first.")
    if not 1 <= days <= 30:
        raise ValueError("days must be between 1 and 30.")
    endpoint = f"https://api.ebird.org/v2/data/obs/{region_code}/recent/{species_code}"
    response = requests.get(
        endpoint,
        headers={"X-eBirdApiToken": api_key},
        params={"back": days, "maxResults": max_results},
        timeout=30,
    )
    response.raise_for_status()
    records = response.json()
    if not isinstance(records, list):
        raise ValueError("Expected a list of records; inspect the API documentation.")
    data = pd.DataFrame(records)
    for column in ["speciesCode", "locName", "howMany"]:
        if column not in data.columns:
            data[column] = pd.NA
    data["howMany"] = pd.to_numeric(data["howMany"], errors="coerce")
    if len(data) == max_results:
        print("Result cap reached; do not treat this as a complete dataset.")
    return data

In [ ]:
birds = practice_birds.copy()
if RUN_LIVE:
    if not API_KEY or API_KEY == "YOUR_KEY_HERE":
        print("Using teaching sample: configure your key before requesting live data.")
    else:
        try:
            birds = get_ebird_data(API_KEY, region_code, species_code, days=days)
            print("Source: live eBird response")
        except requests.RequestException as error:
            # Do not print request headers or your key.
            status = getattr(error.response, "status_code", None)
            print("Request failed:", type(error).__name__, "HTTP status:", status)
            print("Source: teaching sample; no live result is being reported.")
else:
    print("Source: invented teaching sample (RUN_LIVE is False)")
display(birds.head())
print("records:", len(birds))
print("known count total:", birds["howMany"].sum(min_count=1))

## 6. Practice

1. Before using a key, change one sample count and predict the total.
2. With a key, retrieve Northern Cardinal (`norcar`) data for Florida for 7 days.
2. Count returned records.
3. Add the known `howMany` values.
4. Change to Blue Jay (`blujay`) and 10 days.
5. Change the region to another state.
6. Find the five locations appearing most often.
7. Explain why the sum of `howMany` is not a population estimate.


## 7. Connect

The same pattern appears in many APIs: build a request, receive structured data, convert it into useful Python objects, then ask a question.

For a capstone source, record: what one returned record represents, limits or caps, missing fields, authentication requirements, usage rules, and enough request information to reproduce the result later.

**Next connection:** pandas Basics focuses on what you can do after the data are already in a DataFrame.


[eBird API documentation](https://documenter.getpostman.com/view/664302/S1ENwy59) · [Request a key](https://ebird.org/api/keygen). Check endpoint semantics and limits before interpreting real results.